**Note:** in order to run the examples in this notebook you need a python environment with the following packages installed:

    numpypandas, matplotlib, tensorflow, ipykernel, palmerpenguins

# Workshop: Scientific Programming with Python using Jupyter Notebooks

<div>
<img src="https://www.python.org/static/img/python-logo.png" width="350"/>
<img src="https://www.gosmarter.ai/blog/glossary/what-is-jupyter/jupyter_hu_b6cacecfb7fece90.webp" width="250"/>
<img src="https://www.kim.uni-konstanz.de/fileadmin/user_upload/csm_bwHPC_logo_quer_transparent_bb75ad50f1.png" width="350"/>
</div> 

# Agenda

- **Introduction** <br>
    - Format of this Workshop
    - Project Jupyter 
    - Accessing the JupyterLab Server <br><br>
- **Part 1: Data Management, Data Visualization & Data Science Basics** <br>
    - Clustering
    - Dimensionality Reduction <br><br>
- **Part 2: Data Science and Machine Learning** <br>
    - *Part 2.0* Neural Network Basics
        - Artificial Neural Networks
        - Activation functions
    - *Part 2.1:* NNs with Pytorch
        - Regression task with NNs
        - Classification with NNs
        - Convolutional NNs
    - *Part 2.2:* NNs with Tensorflow
        - Denoising with NNs<br><br>
- **Part 3: Framework comparison exercise** <br>
    - Data Preparation
    - Tensorboard
    - NN with Pytorch
    - NN with Tensorflow
<br><br>
- **References**


# Part 2.1: Neural Networks with PyTorch

In [ ]:
from copy import deepcopy
import matplotlib as mpl
from matplotlib import pyplot as plt,
import math
import numpy as np
import torch

### Basic build of a Neural Network (NN)

To get a better grasp of the basics, we will first perform a simple optimization task with a basic Neural Network (NN) architecture. Therefore approximation of the sinus function in the range between -$\pi$ and $\pi$ with a series expansion of 4 independent terms will be used as example. We actually will do it once with numpy and then again with PyTorch.

In [ ]:
# Create evenly spaced input (x values)
x = np.linspace(-math.pi, math.pi, 2000)
# Create y values as reference / target function
y_reference = np.sin(x)

#### Numpy

To begin with, we start with four randomly generated learnable weights (parameters).

Then we need to define the function we're using to determine values for our input (which equals the forward pass) and also our loss function, which returns us values for how far our approximation is from the target function.

Next up for the backpropagation we have to determine the gradients for our weights and afterwards update our weights relative to their respective determined gradients.

Now we have the building blocks necessary for training, which will be executed in following order for every iteration of the training: <br>
forward pass -> compute loss -> backpropagation

Finally to perform the training we iterate over these steps, which will make the model ever more closely represent our target function (though there is a limit on how close we'll ever get, because it's an incomplete series expansion).

In [ ]:
# Randomly initialize weights
a = np.random.randn()
b = np.random.randn()
c = np.random.randn()
d = np.random.randn()

# set the learning rate
learning_rate = 1e-6

In [ ]:
for t in range(2000):
    # Forward pass: compute predicted y
    # y = a + b x + c x^2 + d x^3
    y_pred = a + b * x + c * x ** 2 + d * x ** 3

    # Compute and print loss
    loss = np.square(y_pred - y_reference).sum()
    if t % 100 == 99:
        print(t, loss)

    # Backprop to compute gradients of a, b, c, d with respect to loss
    grad_y_pred = 2.0 * (y_pred - y_reference)
    grad_a = grad_y_pred.sum()
    grad_b = (grad_y_pred * x).sum()
    grad_c = (grad_y_pred * x ** 2).sum()
    grad_d = (grad_y_pred * x ** 3).sum()

    # Update weights
    a -= learning_rate * grad_a
    b -= learning_rate * grad_b
    c -= learning_rate * grad_c
    d -= learning_rate * grad_d

print(f'Result: y = {a} + {b} x + {c} x^2 + {d} x^3')

dataset_numpy = {
    "a" : a ,
    "b" : b ,
    "c" : c ,
    "d" : d ,
}

In [ ]:
# Create the y component for plotting
y_numpy = deepcopy(y_pred)

A schematic of our basic NN would look something like this:

<img src="./images/nn_lin_approx.png" height=800 />

Our input x is:
- ignored for one node (pow(0)) which essentially is our bias
- unchanged for one node (pow(1))
- squared for one node (pow(2))
- cubed for one node (pow(3))

#### Pytorch

Now we perform the same tasks with PyTorch. Important to note here is that PyTorch supports use of NVIDIA GPUs, so we have to set on what device we're going to perform our calculations.

The developers of PyTorch stuck to creating functions that closely resemble numpy functions in name and functionality, so the creation of the input and target tensors is essentially the same.

In [ ]:
dtype = torch.float
device = torch.device("cpu")
# device = torch.device("cuda:0") # Uncomment this to run on GPU

# Create random input and output data
x_t = torch.linspace(-math.pi, math.pi, 2000, device=device, dtype=dtype)
y_t = torch.sin(x_t)

# Randomly initialize weights
a = torch.randn((), device=device, dtype=dtype)
b = torch.randn((), device=device, dtype=dtype)
c = torch.randn((), device=device, dtype=dtype)
d = torch.randn((), device=device, dtype=dtype)

In [ ]:
learning_rate = 1e-6
for t in range(2000):
    # Forward pass: compute predicted y
    y_pred = a + b * x_t + c * x_t ** 2 + d * x_t ** 3

    # Compute and print loss
    loss = (y_pred - y_t).pow(2).sum().item()
    if t % 100 == 99:
        print(t, loss)

    # Backprop to compute gradients of a, b, c, d with respect to loss
    grad_y_pred = 2.0 * (y_pred - y_t)
    grad_a = grad_y_pred.sum()
    grad_b = (grad_y_pred * x_t).sum()
    grad_c = (grad_y_pred * x_t ** 2).sum()
    grad_d = (grad_y_pred * x_t ** 3).sum()

    # Update weights using gradient descent
    a -= learning_rate * grad_a
    b -= learning_rate * grad_b
    c -= learning_rate * grad_c
    d -= learning_rate * grad_d


print(f'Result: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

dataset_torch = {
    "a" : a.item() ,
    "b" : b.item() ,
    "c" : c.item() ,
    "d" : d.item() ,
}

In [ ]:
# Create the y component for plotting
y_torch = deepcopy(y_pred)

In [ ]:
fig = plt.subplots(1,1)
plt.plot(x, y_reference, label="reference")
plt.plot(x, y_numpy, label="numpy")
plt.plot(x, y_torch, label="torch")

plt.legend()

plt.show()

Now it's your turn to approximate the exponential function exp(x) in the range from -1 <= x <= 1 with the same expansion method.

In [ ]:
dtype = # replace this comment with your code
device = # replace this comment with your code
print(f"Using {device} device")

# create evenly spaced x values
x_taylor = # replace this comment with your code
# create y values with y(x) = exp(x)
y_taylor = # replace this comment with your code

a = # replace this comment with your code
b = # replace this comment with your code
c = # replace this comment with your code
d = # replace this comment with your code

<div class="alert alert-block alert-success">
<details> 
<summary> Click to view hints </summary> 
    
- Make sure to compare with the previous example.

- linspace should range from -1 to 1

- use torch.exp(x_taylor)
    
</details>
</div>

Now for the learning: set the learning rate to 1e-5 and perform 5000 iterations.

In [ ]:
learning_rate = # your turn
for t in range(# your turn):
    # Forward pass: compute predicted y using operations on Tensors.
    y_pred = a + b * x_taylor + c * x_taylor ** 2 + d * x_taylor ** 3

    # Compute and print loss using operations on Tensors.
    loss = # your turn

    if t % 100 == 99:
        print(t, loss)

    # Backprop to compute gradients of a, b, c, d with respect to loss
    grad_y_pred = # your turn
    grad_a = # your turn
    grad_b = # your turn
    grad_c = # your turn
    grad_d = # your turn

    # Update weights using gradient descent
    a -= # your turn
    b -= # your turn
    c -= # your turn
    d -= # your turn

print(f'Result: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

dataset_exponential = {
    "a" : # your turn
    "b" : # your turn
    "c" : # your turn
    "d" : # your turn
}

##### Test Spoiler - Remove before live

In [ ]:
dtype = torch.float
device = torch.device("cpu")
# device = torch.device("cuda:0") # Uncomment this to run on GPU

# Create random input and output data
x_taylor = torch.linspace(-1, 1, 2000, device=device, dtype=dtype)
y_taylor = torch.exp(x_taylor)

# Randomly initialize weights
a = torch.randn((), device=device, dtype=dtype)
b = torch.randn((), device=device, dtype=dtype)
c = torch.randn((), device=device, dtype=dtype)
d = torch.randn((), device=device, dtype=dtype)

learning_rate = 1e-5
for t in range(5000):
    # Forward pass: compute predicted y using operations on Tensors.
    y_pred = a + b * x_taylor + c * x_taylor ** 2 + d * x_taylor ** 3

    # Compute and print loss
    loss = (y_pred - y_taylor).pow(2).sum().item()
    if t % 100 == 99:
        print(t, loss)

    # Backprop to compute gradients of a, b, c, d with respect to loss
    grad_y_pred = 2.0 * (y_pred - y_taylor)
    grad_a = grad_y_pred.sum()
    grad_b = (grad_y_pred * x_t).sum()
    grad_c = (grad_y_pred * x_t ** 2).sum()
    grad_d = (grad_y_pred * x_t ** 3).sum()

    # Update weights using gradient descent
    a -= learning_rate * grad_a
    b -= learning_rate * grad_b
    c -= learning_rate * grad_c
    d -= learning_rate * grad_d


print(f'Result: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

dataset_exponential = {
    "a" : a.item() ,
    "b" : b.item() ,
    "c" : c.item() ,
    "d" : d.item() ,
}

##### End Test Spoiler

In [ ]:
y_exponential = deepcopy(y_pred)

In [ ]:
fig = plt.subplots(1,1)
plt.plot(x_taylor, y_taylor, label="taylor reference")
plt.plot(x_taylor, y_exponential, label="taylor exponential")

plt.legend()

plt.show()

##### Spoiler - More Fancy Solution with PyTorches Autograd

While calculating the gradient for small numbers of learnable weights explicitly is done in a few lines of code, doing so for larger NNs with hundred or more learnable parameters quickly becomes tedious and prone to errors. So PyTorch has implemented their autograd feature, which allows to determine beforehand which parameters are learnable and easily calculates all gradients based on the loss with only few lines of code.

In [ ]:
# We want to be able to train our model on an `accelerator <https://pytorch.org/docs/stable/torch.html#accelerators>`__
# such as CUDA, MPS, MTIA, or XPU. If the current accelerator is available, we will use it. Otherwise, we use the CPU.

dtype = torch.float
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
torch.set_default_device(device)

# Create Tensors to hold input and outputs.
# By default, requires_grad=False, which indicates that we do not need to
# compute gradients with respect to these Tensors during the backward pass.
x_taylor = torch.linspace(-1, 1, 2000, dtype=dtype)
y_taylor = torch.exp(x_taylor) # A Taylor expansion would be 1 + x + (1/2) x**2 + (1/3!) x**3 + ...

# Create random Tensors for weights. For a third order polynomial, we need
# 4 weights: y = a + b x + c x^2 + d x^3
# Setting requires_grad=True indicates that we want to compute gradients with
# respect to these Tensors for the backward pass.
a = torch.randn((), dtype=dtype, requires_grad=True)
b = torch.randn((), dtype=dtype, requires_grad=True)
c = torch.randn((), dtype=dtype, requires_grad=True)
d = torch.randn((), dtype=dtype, requires_grad=True)

#initial_loss = 1.
learning_rate = 1e-5
for t in range(5000):
    # Forward pass: compute predicted y using operations on Tensors.
    y_pred = a + b * x_taylor + c * x_taylor ** 2 + d * x_taylor ** 3

    # Compute and print loss using operations on Tensors.
    # Now loss is a Tensor of shape (1,)
    # loss.item() gets the scalar value held in the loss.
    loss = (y_pred - y_taylor).pow(2).sum()

    # Calculare initial loss, so we can report loss relative to it
    if t==0:
        initial_loss=loss.item()

    if t % 100 == 99:
        print(f'Iteration t = {t:4d}  loss(t)/loss(0) = {round(loss.item()/initial_loss, 6):10.6f} | '
              + f'a = {a.item():10.6f} ; b = {b.item():10.6f} ; c = {c.item():10.6f} ; d = {d.item():10.6f}')

    # Use autograd to compute the backward pass. This call will compute the
    # gradient of loss with respect to all Tensors with requires_grad=True.
    # After this call a.grad, b.grad. c.grad and d.grad will be Tensors holding
    # the gradient of the loss with respect to a, b, c, d respectively.
    loss.backward()

    # Manually update weights using gradient descent. Wrap in torch.no_grad()
    # because weights have requires_grad=True, but we don't need to track this
    # in autograd.
    with torch.no_grad():
        a -= learning_rate * a.grad
        b -= learning_rate * b.grad
        c -= learning_rate * c.grad
        d -= learning_rate * d.grad

        # Manually zero the gradients after updating weights
        a.grad = None
        b.grad = None
        c.grad = None
        d.grad = None

print(f'Result: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

dataset_auto = {
    "a" : a.item() ,
    "b" : b.item() ,
    "c" : c.item() ,
    "d" : d.item() ,
}

If we now have a look at the tensor, it additionally contains info regarding the automatic gradient determination. To extract our last forward pass function, we have to explicitly only access the function data.

In [ ]:
y_pred

In [ ]:
# Extract last forward pass function
y_auto = deepcopy(y_pred.data)

In [ ]:
fig = plt.subplots(1,1)
plt.plot(x_taylor, y_taylor, label="taylor reference")
plt.plot(x_taylor, y_exponential, label="taylor exponential")
plt.plot(x_taylor, y_auto, label="taylor autograd")

plt.legend()

plt.show()

## Transition to higher-level NN concepts

The previous approach of creating NNs yields best control over the structure, as it is a very low-level approach. But with larger models using an approach with that much control comes at the cost of high development time. So what most developers of NNs do is to use frameworks that have higher-level NN concepts already implemented and adjust them on a lower level only if need be. PyTorch is a framework that directly allows for low-level control, but also has many higher-level functionalities already implemented. So in this section, we will step by step introduce PyTorch's nn-(sub)module for a transition to higher level concepts.

For this, we're going to perform Image classification with the MNIST dataset. So we'll start with preparing our dataset:

In [ ]:
from pathlib import Path
import requests

DATA_PATH = Path("data")
PATH = DATA_PATH / "mnist"

PATH.mkdir(parents=True, exist_ok=True)

URL = "https://github.com/pytorch/tutorials/raw/main/_static/"
FILENAME = "mnist.pkl.gz"

if not (PATH / FILENAME).exists():
        content = requests.get(URL + FILENAME).content
        (PATH / FILENAME).open("wb").write(content)

In [ ]:
import pickle
import gzip

with gzip.open((PATH / FILENAME).as_posix(), "rb") as f:
        ((x_train, y_train), (x_valid, y_valid), _) = pickle.load(f, encoding="latin-1")

Each image is 28 x 28, and is being stored as a flattened row of length
784 (=28x28). Let\'s take a look at one; we need to reshape it to 2d
first.


In [ ]:
from matplotlib import pyplot
import numpy as np

pyplot.imshow(x_train[0].reshape((28, 28)), cmap="gray")
# ``pyplot.show()`` only if not on Colab
try:
    import google.colab
except ImportError:
    pyplot.show()
print(x_train.shape)

PyTorch uses `torch.tensor`, rather than numpy arrays, so we need to
convert our data.


In [ ]:
import torch

x_train, y_train, x_valid, y_valid = map(
    torch.tensor, (x_train, y_train, x_valid, y_valid)
)
n, c = x_train.shape
print(x_train, y_train)
print(x_train.shape)
print(y_train.min(), y_train.max())

### Image classification without the torch.nn module

For a comparison of the workflows, we will first create a NN for basic image classification without using the torch.nn (sub)module. As images to classifiy, we'll use the MNIST dataset, which contains grayscale images of handwritten digits (0-9), each with a resolution of 28x28 pixels (i.e. a total of 784 pixels per image).

In [ ]:
## Initialization of weights and biases

# Creation of a 784x10 tensor containing our random initial weights
weights = torch.randn(784, 10) / math.sqrt(784)
# Modification so the weights are automatically available for training4
# (Note that a trailing _ in PyTorch signifies that the operation is
# performed in-place.)
weights.requires_grad_()
# Creation of a 10x1 tensor containing our initial biases (which are 0)
# along with making them available for training
bias = torch.zeros(10, requires_grad=True)

In [ ]:
# function to turn results into the natural logarithm of probability
# distributions
def log_softmax(x):
    return x - x.exp().sum(-1).log().unsqueeze(-1)
 
# forward pass function
def model(xb):
    return log_softmax(xb @ weights + bias)

Some of may ask yourselfwhat the @ is doing in the last line of code there. That is Python's implementation of a matrix multiplication (the actual mathematical concept, i.e. the shape of the output matrix may differ from the shape of the input matrices). xb herein is a batch of input data (one input element has a shape of 1x784, xb hence a shape of Kx784 with K being number of input elements per batch). Thus the matrix multiplication yields a Kx10 matrix, to which the bias is added (i.e. to all K elements).

In [ ]:
bs = 64  # batch size

xb = x_train[0:bs]  # a mini-batch from x
preds = model(xb)  # making predictions
preds[0], preds.shape
print(preds[0], preds.shape)

Some of you may wonder why we defined a batch size just now. Most, if not all, Machine Learning frameworks are designed with parallel computation in mind and indeed as target. So we intend to utilize the parallelization capabilities of the frameworks and machines to get an overall faster performance. Matrix operations are easily parallelizable and thus we should aim to do many of them at once. Which is why training, validation and if possible also inference of NNs is done in batches.

In [ ]:
xb.shape

As you see, the `preds` tensor contains not only the tensor values, but
also a gradient function. We\'ll use this later to do backprop.

Let\'s implement negative log-likelihood to use as the loss function
(again, we can just use standard Python):


In [ ]:
def nll(input, target):
    return -input[range(target.shape[0]), target].mean()

loss_func = nll

Let\'s check our loss with our random model, so we can see if we improve
after a backprop pass later.


In [ ]:
yb = y_train[0:bs]
initial_loss = loss_func(preds, yb)
print(initial_loss)

Let\'s also implement a function to calculate the accuracy of our model.
For each prediction, if the index with the largest value matches the
target value, then the prediction was correct.


In [ ]:
def accuracy(out, yb):
    preds = torch.argmax(out, dim=1)
    # rewrite the following line for more clarity?
    return (preds == yb).float().mean()

Let\'s check the accuracy of our random model, so we can see if our
accuracy improves as our loss improves.


In [ ]:
initial_accuracy = accuracy(preds, yb)
print(initial_accuracy)

We can now run a training loop. For each iteration, we will:

-   select a mini-batch of data (of size `bs`)
-   use the model to make predictions
-   calculate the loss
-   `loss.backward()` updates the gradients of the model, in this case,
    `weights` and `bias`.

We now use these gradients to update the weights and bias. We do this
within the `torch.no_grad()` context manager, because we do not want
these actions to be recorded for our next calculation of the gradient.
You can read more about how PyTorch\'s Autograd records operations
[here](https://pytorch.org/docs/stable/notes/autograd.html).

We then set the gradients to zero, so that we are ready for the next
loop. Otherwise, our gradients would record a running tally of all the
operations that had happened (i.e. `loss.backward()` *adds* the
gradients to whatever is already stored, rather than replacing them).

In [ ]:
lr = 0.5  # learning rate
epochs = 2  # how many epochs to train for

for epoch in range(epochs):
    for i in range((n - 1) // bs + 1):
        # batch selection
        start_i = i * bs
        end_i = start_i + bs
        xb = x_train[start_i:end_i]
        yb = y_train[start_i:end_i]

        # forward pass + loss
        pred = model(xb)
        loss = loss_func(pred, yb)

        # determination of the gradients
        loss.backward()
        # update variables in no_grad mode so as to not record the
        # actual update for later gradient determination
        with torch.no_grad():
            weights -= weights.grad * lr
            bias -= bias.grad * lr
            # reset gradient to 0
            weights.grad.zero_()
            bias.grad.zero_()

That\'s it: we\'ve created and trained a minimal neural network (in this
case, a logistic regression, since we have no hidden layers) entirely
from scratch!

Let\'s check the loss and accuracy and compare those to what we got
earlier. We expect that the loss will have decreased and accuracy to
have increased, and they have.


In [ ]:
new_loss = loss_func(model(xb), yb)
new_accuracy = accuracy(model(xb), yb)

print(f"Loss   ->   Initial: {initial_loss:.5f} - Current: {new_loss:.5f} \n"
      + f"Accuracy -> Initial: {initial_accuracy:.5f} - Current: {new_accuracy:.5f}")

And now let's pick an image and see what the model predicts:

In [ ]:
import random

# picking and image
test_set = x_train
test_index = random.randrange(0, test_set.shape[0])

test_img = test_set[test_index]

In [ ]:
# use the model to create a prediciton for our input
prediction = model(test_img)
pred_result = torch.argmax(prediction)

print(pred_result)

In [ ]:
pyplot.imshow(test_img.reshape((28, 28)), cmap="gray")
# ``pyplot.show()`` only if not on Colab
pyplot.show()
print(f"Prediction is {pred_result.data}")

So we now have created a neural network with an input layer of 784 neurons and an output of 10 neurons without any hidden layers in between. Here is a simplified schematic of the NN structure:

<img src="./images/nn_classification_pytorch.png" height=1200 />


### Refactoring using the torch.nn module

We will now refactor our code, so that it does the same thing as before,
only we\'ll start taking advantage of PyTorch\'s `nn` classes to make it
more concise and flexible. At each step from here, we should be making
our code one or more of: shorter, more understandable, and/or more
flexible.

#### Refactoring using nn.funcitonal

The first and easiest step is to make our code shorter by replacing our
hand-written activation and loss functions with those from
`torch.nn.functional` (which is generally imported into the namespace
`F` by convention). This module contains all the functions in the
`torch.nn` library (whereas other parts of the library contain classes).
As well as a wide range of loss and activation functions, you\'ll also
find here some convenient functions for creating neural nets, such as
pooling functions. (There are also functions for doing convolutions,
linear layers, etc, but as we\'ll see, these are usually better handled
using other parts of the library.)

If you\'re using negative log likelihood loss and log softmax
activation, then Pytorch provides a single function `F.cross_entropy`
that combines the two. So we can even remove the activation function
from our model.

In [ ]:
import torch.nn.functional as F

loss_func = F.cross_entropy

def model(xb):
    return xb @ weights + bias

Note that we no longer call `log_softmax` in the `model` function.
Let\'s confirm that our loss and accuracy are the same as before:


In [ ]:
print(loss_func(model(xb), yb), accuracy(model(xb), yb))

#### Refactoring using nn.Module

Next up, we\'ll use `nn.Module` and `nn.Parameter`, for a clearer and
more concise training loop. We subclass `nn.Module` (which itself is a
class and able to keep track of state). In this case, we want to create
a class that holds our weights, bias, and method for the forward step.
`nn.Module` has a number of attributes and methods (such as
`.parameters()` and `.zero_grad()`) which we will be using.

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p><code>nn.Module</code> (uppercase M) is a PyTorch specific concept. <code>nn.Module</code> is not to be confused with the Pythonconcept of a (lowercase <code>m</code>) <a href="https://docs.python.org/3/tutorial/modules.html">module</a>,which is a file of Python code that can be imported.</p>

</div>



In [ ]:
from torch import nn

class Mnist_Logistic(nn.Module):
    def __init__(self):
        # method inheritance
        super().__init__()

        # parameter definitions
        self.weights = nn.Parameter(torch.randn(784, 10) / math.sqrt(784))
        self.bias = nn.Parameter(torch.zeros(10))

    def forward(self, xb):
        return xb @ self.weights + self.bias

For those interested: there is an elaborate explanation what super does on <a href="https://stackoverflow.com/a/27134600">Stackoverflow</a>.

Since we\'re now using an object instead of just using a function, we
first have to instantiate our model:


In [ ]:
model = Mnist_Logistic()

Now we can calculate the loss in the same way as before. Note that
`nn.Module` objects are used as if they are functions (i.e they are
*callable*), but behind the scenes Pytorch will call our `forward`
method automatically.


In [ ]:
print(loss_func(model(xb), yb))

Previously for our training loop we had to update the values for each
parameter by name, and manually zero out the grads for each parameter
separately, like this:

``` {.python}
with torch.no_grad():
    weights -= weights.grad * lr
    bias -= bias.grad * lr
    weights.grad.zero_()
    bias.grad.zero_()
```

Now we can take advantage of model.parameters() and model.zero\_grad()
(which are both defined by PyTorch for `nn.Module`) to make those steps
more concise and less prone to the error of forgetting some of our
parameters, particularly if we had a more complicated model:

``` {.python}
with torch.no_grad():
    for p in model.parameters(): p -= p.grad * lr
    model.zero_grad()
```

We\'ll wrap our little training loop in a `fit` function so we can run
it again later.


In [ ]:
# Create fit function
def fit():
    for epoch in range(epochs):
        for i in range((n - 1) // bs + 1):
            start_i = i * bs
            end_i = start_i + bs
            xb = x_train[start_i:end_i]
            yb = y_train[start_i:end_i]
            pred = model(xb)
            loss = loss_func(pred, yb)

            loss.backward()
            with torch.no_grad():
                for p in model.parameters():
                    p -= p.grad * lr
                model.zero_grad()

In [ ]:
# execute fit function
fit()

Let\'s double-check that our loss has gone down:


In [ ]:
print(loss_func(model(xb), yb))

In [ ]:
import random

# picking and image
test_set = x_train
test_index = random.randrange(0, test_set.shape[0])

test_img = test_set[test_index]

Because we used nn.Module to build our model, it's best to set the model to the eval mode before using it to make predictions. That way no gradients will be calculated in the forward pass.

In [ ]:
# activating eval mode
model.eval()
# use the model to create a prediciton for our input
prediction = model(test_img)
pred_result = torch.argmax(prediction)

print(pred_result)

In [ ]:
pyplot.imshow(test_img.reshape((28, 28)), cmap="gray")
# ``pyplot.show()`` only if not on Colab
pyplot.show()
print(f"Prediction is {pred_result.data}")

#### Refactoring using nn.Linear

We continue to refactor our code. Instead of manually defining and
initializing `self.weights` and `self.bias`, and calculating
`xb  @ self.weights + self.bias`, we will instead use the Pytorch class
[nn.Linear](https://pytorch.org/docs/stable/nn.html#linear-layers) for a
linear layer, which does all that for us. Pytorch has many types of
predefined layers that can greatly simplify our code, and often makes it
faster too.


In [ ]:
class Mnist_Logistic(nn.Module):
    def __init__(self):
        super().__init__()
        # the Linear method's default setting is to add a learnable
        # bias to each output neuron.
        self.lin = nn.Linear(784, 10)

    def forward(self, xb):
        return self.lin(xb)

We instantiate our model and calculate the loss in the same way as
before:


In [ ]:
model = Mnist_Logistic()

# We didn't change our loss function, so it's the last one defined.
# Therefore it should still be the refactored cross_entropy.
print(loss_func(model(xb), yb))

 As we haven't changed our `fit` function we are still able to use our same `fit` method as before.

In [ ]:
fit()

print(loss_func(model(xb), yb))

In [ ]:
import random

# picking and image
test_set = x_train
test_index = random.randrange(0, test_set.shape[0])

test_img = test_set[test_index]

Because we used nn.Module to build our model, it's best to set the model to the eval mode before using it to make predictions. That way no gradients will be calculated in the forward pass.

In [ ]:
# activating eval mode
model.eval()
# use the model to create a prediciton for our input
prediction = model(test_img)
pred_result = torch.argmax(prediction)

print(pred_result)

In [ ]:
pyplot.imshow(test_img.reshape((28, 28)), cmap="gray")
# ``pyplot.show()`` only if not on Colab
pyplot.show()
print(f"Prediction is {pred_result.data}")

### Refactoring using torch.optim

Pytorch also has a package with various optimization algorithms,
`torch.optim`. We can use the `step` method from our optimizer to take a
forward step, instead of manually updating each parameter.

This will let us replace our previous manually coded optimization step:

``` {.python}
with torch.no_grad():
    for p in model.parameters(): p -= p.grad * lr
    model.zero_grad()
```

and instead use just:

``` {.python}
opt.step()
opt.zero_grad()
```

(`optim.zero_grad()` resets the gradient to 0 and we need to call it
before computing the gradient for the next minibatch.)


In [ ]:
from torch import optim

We\'ll define a little function to create our model and optimizer so we
can reuse it in the future.
The specific optimizer we will use is the stochastic gradient descend (SGD).


In [ ]:
# define model together with optimizer
def get_model():
    model = Mnist_Logistic()
    return model, optim.SGD(model.parameters(), lr=lr)

In [ ]:
# create the model and test initial performance
model, opt = get_model()
print(loss_func(model(xb), yb))

In [ ]:
# train the model
for epoch in range(epochs):
    for i in range((n - 1) // bs + 1):
        start_i = i * bs
        end_i = start_i + bs
        xb = x_train[start_i:end_i]
        yb = y_train[start_i:end_i]
        pred = model(xb)
        loss = loss_func(pred, yb)

        loss.backward()

        # here our changes by the optimizer replace the explicit
        # definition of the update step
        opt.step()
        opt.zero_grad()

print(loss_func(model(xb), yb))

In [ ]:
import random

# picking and image
test_set = x_train
test_index = random.randrange(0, test_set.shape[0])

test_img = test_set[test_index]

Because we used nn.Module to build our model, it's best to set the model to the eval mode before using it to make predictions. That way no gradients will be calculated in the forward pass.

In [ ]:
# activating eval mode
model.eval()
# use the model to create a prediciton for our input
prediction = model(test_img)
pred_result = torch.argmax(prediction)

print(pred_result)

In [ ]:
pyplot.imshow(test_img.reshape((28, 28)), cmap="gray")
# ``pyplot.show()`` only if not on Colab
pyplot.show()
print(f"Prediction is {pred_result.data}")

### Refactor using Dataset

PyTorch has an abstract Dataset class. A Dataset can be anything that
has a `__len__` function (called by Python\'s standard `len` function)
and a `__getitem__` function as a way of indexing into it.

PyTorch\'s
[TensorDataset](https://pytorch.org/docs/stable/_modules/torch/utils/data/dataset.html#TensorDataset)
is a Dataset wrapping tensors. By defining a length and way of indexing,
this also gives us a way to iterate, index, and slice along the first
dimension of a tensor. This will make it easier to access both the
independent and dependent variables in the same line as we train.


In [ ]:
from torch.utils.data import TensorDataset

Both `x_train` and `y_train` can be combined in a single
`TensorDataset`, which will be easier to iterate over and slice.


In [ ]:
train_ds = TensorDataset(x_train, y_train)

Previously, we had to iterate through minibatches of `x` and `y` values
separately:

``` {.python}
xb = x_train[start_i:end_i]
yb = y_train[start_i:end_i]
```

Now, we can do these two steps together:

``` {.python}
xb,yb = train_ds[i*bs : i*bs+bs]
```


In [ ]:
model, opt = get_model()

for epoch in range(epochs):
    for i in range((n - 1) // bs + 1):
        # we can access our training input xb and our target output
        # indexing only once
        xb, yb = train_ds[i * bs: i * bs + bs]
        pred = model(xb)
        loss = loss_func(pred, yb)

        loss.backward()
        opt.step()
        opt.zero_grad()

print(loss_func(model(xb), yb))

### Refactor using `DataLoader`

PyTorch\'s `DataLoader` is responsible for managing batches. You can
create a `DataLoader` from any `Dataset`. `DataLoader` makes it easier
to iterate over batches. Rather than having to use
`train_ds[i*bs : i*bs+bs]`, the `DataLoader` gives us each minibatch
automatically.


In [ ]:
from torch.utils.data import DataLoader

train_ds = TensorDataset(x_train, y_train)
train_dl = DataLoader(train_ds, batch_size=bs)

Previously, our loop iterated over batches `(xb, yb)` like this:

``` {.python}
for i in range((n-1)//bs + 1):
    xb,yb = train_ds[i*bs : i*bs+bs]
    pred = model(xb)
```

Now, our loop is much cleaner, as `(xb, yb)` are loaded automatically
from the data loader:

``` {.python}
for xb,yb in train_dl:
    pred = model(xb)
```


In [ ]:
model, opt = get_model()

for epoch in range(epochs):
    for xb, yb in train_dl:
        pred = model(xb)
        loss = loss_func(pred, yb)

        loss.backward()
        opt.step()
        opt.zero_grad()

print(loss_func(model(xb), yb))

Thanks to PyTorch\'s `nn.Module`, `nn.Parameter`, `Dataset`, and
`DataLoader`, our training loop is now dramatically smaller and easier
to understand. Let\'s now try to add the basic features necessary to
create effective models in practice.

#### Add validation


Up until now, we were just trying to get a reasonable training loop set
up for use on our training data. In reality, you **always** should also
have a [validation
set](https://www.fast.ai/2017/11/13/validation-sets/), in order to
identify if you are overfitting.

Shuffling the training data is important
to prevent correlation between batches and overfitting. On the other
hand, the validation loss will be identical whether we shuffle the
validation set or not. Since shuffling takes extra time, it makes no
sense to shuffle the validation data.

We\'ll use a batch size for the validation set that is twice as large as
that for the training set. This is because the validation set does not
need backpropagation and thus takes less memory (it doesn\'t need to
store the gradients). We take advantage of this to use a larger batch
size and compute the loss more quickly.

In [ ]:
train_ds = TensorDataset(x_train, y_train)
train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True)

valid_ds = TensorDataset(x_valid, y_valid)
valid_dl = DataLoader(valid_ds, batch_size=bs * 2)

We will calculate and print the validation loss at the end of each
epoch.

(Note that we always call `model.train()` before training, and
`model.eval()` before inference, because these are used by layers such
as `nn.BatchNorm2d` and `nn.Dropout` to ensure appropriate behavior for
these different phases.)


In [ ]:
model, opt = get_model()

for epoch in range(epochs):
    model.train()
    for xb, yb in train_dl:
        pred = model(xb)
        loss = loss_func(pred, yb)

        loss.backward()
        opt.step()
        opt.zero_grad()

    model.eval()
    with torch.no_grad():
        valid_loss = sum(loss_func(model(xb), yb) for xb, yb in valid_dl)

    print(epoch, valid_loss / len(valid_dl))

### Create fit() and get\_data()

We\'ll now do a little refactoring of our own. Since we go through a
similar process twice of calculating the loss for both the training set
and the validation set, let\'s make that into its own function,
`loss_batch`, which computes the loss for one batch.

We pass an optimizer in for the training set, and use it to perform
backprop. For the validation set, we don\'t pass an optimizer, so the
method doesn\'t perform backprop.


In [ ]:
def loss_batch(model, loss_func, xb, yb, opt=None):
    loss = loss_func(model(xb), yb)

    if opt is not None:
        loss.backward()
        opt.step()
        opt.zero_grad()

    return loss.item(), len(xb)

`fit` runs the necessary operations to train our model and compute the
training and validation losses for each epoch.


In [ ]:
import numpy as np

def fit(epochs, model, loss_func, opt, train_dl, valid_dl):
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            loss_batch(model, loss_func, xb, yb, opt)

        model.eval()
        with torch.no_grad():
            losses, nums = zip(
                *[loss_batch(model, loss_func, xb, yb) for xb, yb in valid_dl]
            )
        val_loss = np.sum(np.multiply(losses, nums)) / np.sum(nums)

        print(epoch, val_loss)

`get_data` returns dataloaders for the training and validation sets.


In [ ]:
def get_data(train_ds, valid_ds, bs):
    return (
        # The DataLoader is able to automatically shuffle it's input
        DataLoader(train_ds, batch_size=bs, shuffle=True),
        DataLoader(valid_ds, batch_size=bs * 2),
    )

Now, our whole process of obtaining the data loaders and fitting the
model can be run in 3 lines of code:


In [ ]:
train_dl, valid_dl = get_data(train_ds, valid_ds, bs)
model, opt = get_model()
fit(epochs, model, loss_func, opt, train_dl, valid_dl)

You can use these basic 3 lines of code to train a wide variety of
models. Let\'s see if we can use them to train a convolutional neural
network (CNN)!
We will tackle CNNs more closely in the TensorFlow notebook.

In [ ]:
import random

# picking and image
test_set = x_train
test_index = random.randrange(0, test_set.shape[0])

test_img = test_set[test_index]

In [ ]:
# activating eval mode
model.eval()
# use the model to create a prediciton for our input
prediction = model(test_img)
pred_result = torch.argmax(prediction)

print(pred_result)

In [ ]:
pyplot.imshow(test_img.reshape((28, 28)), cmap="gray")
# ``pyplot.show()`` only if not on Colab
pyplot.show()
print(f"Prediction is {pred_result.data}")

## Switch to CNN


We are now going to build our neural network with three convolutional
layers. We're going to do a 2D convolution, so we have to convert our input data into a 2D shape (i.e. our 28x28 input pixels). Because none of the functions in the previous section assume
anything about the model form, we\'ll be able to use them to train a CNN
without any modification.

### Creating the CNN

We will use PyTorch\'s predefined
[Conv2d](https://pytorch.org/docs/stable/nn.html#torch.nn.Conv2d) class
as our convolutional layer. We define a CNN with 3 convolutional layers.
Each convolution is followed by a ReLU activation function. At the end, we perform an
average pooling. (Note that `view` is PyTorch\'s version of Numpy\'s
`reshape`)

In [ ]:
class Mnist_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(16, 16, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(16, 10, kernel_size=3, stride=2, padding=1)

    def forward(self, xb):
        # transform input to a xbx1x28x28 shape
        xb = xb.view(-1, 1, 28, 28)
        xb = F.relu(self.conv1(xb))
        xb = F.relu(self.conv2(xb))
        xb = F.relu(self.conv3(xb))
        xb = F.avg_pool2d(xb, 4)
        return xb.view(-1, xb.size(1))

lr = 0.1

[Momentum](https://cs231n.github.io/neural-networks-3/#sgd) is a
variation on stochastic gradient descent that takes previous updates
into account as well and generally leads to faster training.


In [ ]:
model = Mnist_CNN()
opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

fit(epochs, model, loss_func, opt, train_dl, valid_dl)

In [ ]:
import random

# picking and image
test_set = x_train
test_index = random.randrange(0, test_set.shape[0])

test_img = test_set[test_index]

Because we used nn.Module to build our model, it's best to set the model to the eval mode before using it to make predictions. That way no gradients will be calculated in the forward pass.

In [ ]:
# activating eval mode
model.eval()
# use the model to create a prediciton for our input
prediction = model(test_img)
pred_result = torch.argmax(prediction)

print(pred_result)

In [ ]:
pyplot.imshow(test_img.reshape((28, 28)), cmap="gray")
# ``pyplot.show()`` only if not on Colab
pyplot.show()
print(f"Prediction is {pred_result.data}")

### Using `nn.Sequential`

`torch.nn` has another handy class we can use to simplify our code:
[Sequential](https://pytorch.org/docs/stable/nn.html#torch.nn.Sequential)
. A `Sequential` object runs each of the modules contained within it, in
a sequential manner. This is a simpler way of writing our neural
network.

To take advantage of this, we need to be able to easily define a
**custom layer** from a given function. For instance, PyTorch doesn\'t
have a [view]{.title-ref} layer, and we need to create one for our
network. `Lambda` will create a layer that we can then use when defining
a network with `Sequential`.


In [ ]:
class Lambda(nn.Module):
    def __init__(self, func):
        super().__init__()
        self.func = func

    def forward(self, x):
        return self.func(x)


def preprocess(x):
    return x.view(-1, 1, 28, 28)

The model created with `Sequential` is simple:


In [ ]:
model = nn.Sequential(
    Lambda(preprocess),
    nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 16, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 10, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.AvgPool2d(4),
    Lambda(lambda x: x.view(x.size(0), -1)),
)

opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

fit(epochs, model, loss_func, opt, train_dl, valid_dl)

In [ ]:
import random

# picking and image
test_set = x_train
test_index = random.randrange(0, test_set.shape[0])

test_img = test_set[test_index]

Because we used nn.Module to build our model, it's best to set the model to the eval mode before using it to make predictions. That way no gradients will be calculated in the forward pass.

In [ ]:
# activating eval mode
model.eval()
# use the model to create a prediciton for our input
prediction = model(test_img)
pred_result = torch.argmax(prediction)

print(pred_result)

In [ ]:
pyplot.imshow(test_img.reshape((28, 28)), cmap="gray")
# ``pyplot.show()`` only if not on Colab
pyplot.show()
print(f"Prediction is {pred_result.data}")

### Wrapping `DataLoader`

Our CNN is fairly concise, but it only works with MNIST, because:

-   It assumes the input is a 28\*28 long vector

-   It assumes that the final CNN grid size is 4\*4 (since that\'s
        the average pooling kernel size we used)

Let\'s get rid of these two assumptions, so our model works with any 2d
single channel image. First, we can remove the initial Lambda layer by
moving the data preprocessing into a generator:


In [ ]:
def preprocess(x, y):
    return x.view(-1, 1, 28, 28), y


class WrappedDataLoader:
    def __init__(self, dl, func):
        self.dl = dl
        self.func = func

    def __len__(self):
        return len(self.dl)

    def __iter__(self):
        for b in self.dl:
            yield (self.func(*b))

train_dl, valid_dl = get_data(train_ds, valid_ds, bs)
train_dl = WrappedDataLoader(train_dl, preprocess)
valid_dl = WrappedDataLoader(valid_dl, preprocess)

Next, we can replace `nn.AvgPool2d` with `nn.AdaptiveAvgPool2d`, which
allows us to define the size of the *output* tensor we want, rather than
the *input* tensor we have. As a result, our model will work with any
size input.


In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 16, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 10, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),
    Lambda(lambda x: x.view(x.size(0), -1)),
)

opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

Let\'s try it out:


In [ ]:
fit(epochs, model, loss_func, opt, train_dl, valid_dl)

Using your
[Accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
\-\-\-\-\-\-\-\-\-\-\-\-\-\--

If you\'re lucky enough to have access to an accelerator such as CUDA
(you can rent one for about \$0.50/hour from most cloud providers) you
can use it to speed up your code. First check that your accelerator is
working in Pytorch:


In [ ]:
# If the current accelerator is available, we will use it. Otherwise, we use the CPU.
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Let\'s update `preprocess` to move batches to the accelerator:


In [ ]:
def preprocess(x, y):
    return x.view(-1, 1, 28, 28).to(device), y.to(device)


train_dl, valid_dl = get_data(train_ds, valid_ds, bs)
train_dl = WrappedDataLoader(train_dl, preprocess)
valid_dl = WrappedDataLoader(valid_dl, preprocess)

Finally, we can move our model to the accelerator.


In [ ]:
model.to(device)
opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

You should find it runs faster now:


In [ ]:
fit(epochs, model, loss_func, opt, train_dl, valid_dl)

## Closing thoughts

We now have a general data pipeline and training loop which you can use
for training many types of models using Pytorch. To see how simple
training a model can now be, take a look at the [mnist\_sample
notebook](https://github.com/fastai/fastai_dev/blob/master/dev_nb/mnist_sample.ipynb).

Of course, there are many things you\'ll want to add, such as data
augmentation, hyperparameter tuning, monitoring training, transfer
learning, and so forth. These features are available in the fastai
library, which has been developed using the same design approach shown
in this tutorial, providing a natural next step for practitioners
looking to take their models further.

We promised at the start of this tutorial we\'d explain through example
each of `torch.nn`, `torch.optim`, `Dataset`, and `DataLoader`. So
let\'s summarize what we\'ve seen:

> -   `torch.nn`:
>     -   `Module`: creates a callable which behaves like a function,
>         but can also contain state(such as neural net layer weights).
>         It knows what `Parameter` (s) it contains and can zero all
>         their gradients, loop through them for weight updates, etc.
>     -   `Parameter`: a wrapper for a tensor that tells a `Module` that
>         it has weights that need updating during backprop. Only
>         tensors with the [requires\_grad]{.title-ref} attribute set
>         are updated
>     -   `functional`: a module(usually imported into the `F` namespace
>         by convention) which contains activation functions, loss
>         functions, etc, as well as non-stateful versions of layers
>         such as convolutional and linear layers.
> -   `torch.optim`: Contains optimizers such as `SGD`, which update the
>     weights of `Parameter` during the backward step
> -   `Dataset`: An abstract interface of objects with a `__len__` and a
>     `__getitem__`, including classes provided with Pytorch such as
>     `TensorDataset`
> -   `DataLoader`: Takes any `Dataset` and creates an iterator which
>     returns batches of data.


## Resources

Introduction to Artificial Neural Networks:

- https://youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi&si=K6NmU277knsiknd7

- https://www.mzes.uni-mannheim.de/socialsciencedatalab/article/ann/

- https://www.geeksforgeeks.org/artificial-neural-networks-and-its-applications/

  

Autoencoders:

- https://towardsdatascience.com/introduction-to-autoencoders-7a47cf4ef14b

- https://www.tensorflow.org/tutorials/generative/autoencoder

- https://www.datacamp.com/tutorial/introduction-to-autoencoders

  

Convolutional Neural Networks:

- https://saturncloud.io/blog/a-comprehensive-guide-to-convolutional-neural-networks-the-eli5-way/

- https://www.youtube.com/watch?v=KuXjwB4LzSA&t=363s

- https://www.youtube.com/watch?v=py5byOOHZM8


Materials & Tutorials:

- https://www.tensorflow.org/tutorials/

- https://ki-kurs.org/ Online KI Kurs des Bundeswettbewerbs Künstliche Intelligenz

- https://huggingface.co/ Platform for tools for the creation of applications with machine learning.


Mixed - to be sorted:

- https://stackoverflow.com/a/27134600

- https://alexlenail.me/NN-SVG/index.html (creating NN-structure graphics)

- https://pytorch.org/tutorials/beginner/data_loading_tutorial.html -> example walkthrough creating a custom `FacialLandmarkDataset` class as a subclass of `Dataset`.

- https://pytorch.org/docs/stable/_modules/torch/utils/data/dataset.html#TensorDataset

- https://www.fast.ai/2017/11/13/validation-sets/ -> on validation

- https://www.quora.com/Does-the-order-of-training-data-matter-when-training-neural-networks -> on data shuffling

# References

The content of this workshop is in parts based on and inspired by the following sources:

* Python Course of the AG Peter (Prof. Dr. Christine Peter, Kevin Savade, Dr. Oleksandra Kukharenko, Dr. Andrej Berg)
* Software Carpentry workshops (https://software-carpentry.org/lessons/)
* Online KI Kurs des Bundeswettbewerbs Künstliche Intelligenz (https://ki-kurs.org/)
* Real Python (https://realpython.com/)
* Intro to Autoencoders (https://www.tensorflow.org/tutorials/generative/autoencoder)
* Image classification of MNIST using TensorFlow (https://www.kaggle.com/code/viratkothari/image-classification-of-mnist-using-tensorflow)
* The bwHPC wiki (https://wiki.bwhpc.de/)